<a href="https://colab.research.google.com/github/kcymae/Project-/blob/main/Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IEDB Database**

The [*IEDB Database*](https://www.iedb.org/) established in 2004 as a free resource to house experimental data related to adaptive immune epitopes and to also provide leading-edge epitope analysis and prediction tools

## **Install Libraries**




In [26]:
pip install iedb

In [27]:
import iedb

# Send POST request to MHC class I peptide binding prediction tool:
mhci_res = iedb.query_mhci_binding(method="recommended", sequence="ARFTGIKTA", allele="HLA-A*02:01", length=8)

# Send POST request to MHC class II peptide binding prediction tool:
mhcii_res = iedb.query_mhcii_binding(method="nn_align", sequence="SLYNTVATLYCVHQRIDV", allele="HLA-DRB1*01:01", length=None)

# Send POST request to T-cell epitope prediction tool:
tcell_res = iedb.query_tcell_epitope(method="smm", sequence="SLYNTVATLYCVHQRIDV", allele="HLA-A*01:01", length=9, proteasome='immuno')

# Send POST request to peptide prediction tool:
pep_res = iedb.query_peptide_prediction(method="mhcnp", sequence="SLYNTVATLYCVHQRIDV", allele="HLA-A*02:01", length=9)

# Send POST request to B-cell epitope prediction tool:
bcell_res = iedb.query_bcell_epitope(method="Emini", sequence="VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTE", window_size=9)


## **Selection of Biological Target Protein**

A viral antigenic protein was selected as the target for epitope prediction. The spike glycoprotein of `SARS-CoV-2` was chosen due to its immunogenic relevance and availability of sequence data

The protein sequence of the SARS-CoV-2 spike glycoprotein (Accession ID: P0DTC2, Gene: S) was retrieved in FASTA format from the UniProt database, representing the Wuhan-Hu-1 isolate (the reference proteome)



In [28]:
#Import the urlretrieve function
from urllib.request import urlretrieve

#Save the looooong web address (url) of our data in a python string
genome_url = 'https://raw.githubusercontent.com/kcymae/Project-/refs/heads/main/P0DTC2_SPIKE_SARS2%20Spike%20glycopr.fasta'

#Set the name of the file where we want to save the data
genome_file_name = 'P0DTC2_SPIKE_SARS2 Spike glycopr.fasta'

#Download the data
urlretrieve(genome_url, genome_file_name)


('P0DTC2_SPIKE_SARS2 Spike glycopr.fasta',
 <http.client.HTTPMessage at 0x782c5ae93770>)

In [29]:
infile = open('P0DTC2_SPIKE_SARS2 Spike glycopr.fasta')
for line in infile:
    print(line)

>sp|P0DTC2|SPIKE_SARS2 Spike glycoprotein OS=Severe acute respiratory syndrome coronavirus 2 OX=2697049 GN=S PE=1 SV=1

MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFS

NVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIV

NNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLE

GKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQT

LLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETK

CTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISN

CVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIAD

YNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPC

NGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVN

FNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITP

GTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSY

ECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTI

SVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQE

VFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDC

LGDIAARDLICA

## **Generation of Candidate Peptides**



In [30]:
! pip install biopython
from Bio import SeqIO

Read the FASTA file

In [31]:
fasta_file ="P0DTC2_SPIKE_SARS2 Spike glycopr.fasta"
record = SeqIO.read(fasta_file, "fasta")
protein_sequence = str(record.seq)

print("Sequence length:", len(protein_sequence))
print(protein_sequence[:50])

Sequence length: 1273
MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHS


Generate overlapping peptides

In [32]:
def generate_peptides(sequence, length=9):
    peptides = []
    for i in range(len(sequence) - length + 1):
        peptide = sequence[i:i+length]
        peptides.append(peptide)
    return peptides

peptides = generate_peptides(protein_sequence, length=9)

print("Total peptides generated:", len(peptides))
print(peptides[:10])

Total peptides generated: 1265
['MFVFLVLLP', 'FVFLVLLPL', 'VFLVLLPLV', 'FLVLLPLVS', 'LVLLPLVSS', 'VLLPLVSSQ', 'LLPLVSSQC', 'LPLVSSQCV', 'PLVSSQCVN', 'LVSSQCVNL']


## **Prediction of MHC Binding Affinity**

Prediction of MHC Class I Binding Affinity Using IEDB

In [33]:
results = []

for pep in peptides[:50]:
    try:
        res = iedb.query_mhci_binding(
            method="recommended",
            sequence=pep,
            allele="HLA-A*02:01"
        )

        for r in res:
            results.append({
                "peptide": pep,
                "allele": r.get("allele"),
                "ic50": r.get("ic50"),
                "rank": r.get("percentile_rank")
            })
    except:
        continue

## **Compilation of Predicted Epitope Dataset**

Convert to Dataframe

In [40]:
import pandas as pd

df = pd.DataFrame(results)
df.head()

""


In [35]:
df.to_csv("epitope_prediction_raw.csv", index=False)